In [0]:
dbutils.widgets.text("Incremental_flag",'0')

In [0]:
Incremental_flag= dbutils.widgets.get("Incremental_flag")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

##Getting silver Data

In [0]:

df = spark.read.format('parquet').load('abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data/')
        

## fetching relative columns

In [0]:
df_src = spark.sql(''' 
SELECT DISTINCT (Dealer_ID) as Dealer_ID, DealerName from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data`
''')

#Getting schema of source df with empty

### dim_model sink initial and incremental load

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_dealer'):
    df_sink= spark.sql('''
        SELECT dim_dealer_key, Dealer_ID, DealerName from cars_data_catalog.gold.dim_dealer
        ''')
else:
    df_sink= spark.sql('''
    SELECT 1 as dim_dealer_key, Dealer_ID, DealerName from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data` 
    where 1=0
    ''')


### Filtering new records vs new records

In [0]:
df_filtered = df_src.join(df_sink, df_src.Dealer_ID==df_sink.Dealer_ID,how='left').select(df_src['Dealer_ID'],df_src['DealerName'], df_sink['dim_dealer_key'])

### df_filter_old

In [0]:
df_filter_old = df_filtered.filter(col('dim_dealer_key').isNotNull())

In [0]:
df_filter_new = df_filtered.filter(col('dim_dealer_key').isNull()).select(df_filtered['Dealer_ID'],df_filtered['DealerName'])


### Create Surrogate Key

**get max value for surrogate key **

In [0]:
if (Incremental_flag== '0'):
    max_value=1
else:
    max_value_df = spark.sql("SELECT MAX(dim_dealer_key)  FROM cars_data_catalog.gold.dim_dealer")
    max_value = max_value_df.collect()[0][0]+1

**create a new surrogate key and add max value in it**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_dealer_key', max_value + monotonically_increasing_id())

### Create Final df = filter_new + filter_old

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_dealer'):
    delta_tbl= DeltaTable.forPath(spark, "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_dealer")

    delta_tbl.alias("trg").merge(df_final.alias("src"), "trg.dim_dealer_key = src.dim_dealer_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format("delta").\
    mode("overwrite")\
    .option("path", "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_dealer")\
    .saveAsTable('cars_data_catalog.gold.dim_dealer')

In [0]:
%sql
select * from cars_data_catalog.gold.dim_dealer;

Dealer_ID,DealerName,dim_dealer_key
DLR0069,Geo Motors,1
DLR0249,Acura Motors,2
DLR0209,Zastava Motors,3
DLR0189,Sunbeam Motors,4
DLR0130,Micro Motors,5
DLR0093,Iso Motors,6
DLR0135,Morgan Motors,7
DLR0170,Saleen Motors,8
DLR0265,Bentley Motors,9
DLR0087,Hyundai Motors,10
